# EDA orientado a decisiones de negocio

**Objetivo de aprendizaje:**
Al finalizar este notebook, el alumno podra:
- Formular preguntas de negocio antes de explorar datos
- Leer un mapa de nulos como riesgo operativo, no como problema tecnico
- Interpretar histogramas, boxplots, scatters y heatmaps desde perspectiva de producto
- Decidir que variables merecen atencion antes de entrenar cualquier modelo

**Conexion con el documento:**
Este notebook acompana la seccion `### 2.0b EDA orientado a decisiones de negocio`.
La teoria del EDA como herramienta de decision esta explicada alli.
Aqui la operacionalizamos con un dataset CRM simulado de la empresa.

> **Antes de seguir:** escribe una pregunta de negocio concreta que quieras
> responder con los datos de este notebook. No es una pregunta tecnica.
> Ejemplo: 'quiero saber si los clientes que abren muchos tickets son los mismos
> que luego cancelan, para priorizar a quien llamar esta semana'.

<details>
<summary>Por que esto importa (desplegar)</summary>

Un EDA sin pregunta previa produce graficas sin interpretacion.
Un EDA con pregunta produce decisiones.
La diferencia no es la tecnica: es el orden mental.

</details>

## Setup

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
IMG = 'images'
os.makedirs(IMG, exist_ok=True)
print('Setup OK')

Setup OK


## 1. Dataset CRM simulado

Simulamos 300 clientes B2B con variables tipicas de un CRM:
segmento, uso mensual de la plataforma, tickets de soporte,
NPS (Net Promoter Score, indice de recomendacion), dias sin login
y variable objetivo: churn (1 = cancelo en los ultimos 6 meses).

Se introducen nulos deliberados en dos columnas para simular
datos reales con informacion incompleta.

In [2]:
# Dataset CRM simulado - 300 clientes B2B
n = 300

segmento = np.random.choice(['SMB', 'Mid-Market', 'Enterprise'],
                            size=n, p=[0.55, 0.30, 0.15])

# Uso mensual: Enterprise usa mas, SMB usa menos
uso_base = {'SMB': 20, 'Mid-Market': 55, 'Enterprise': 120}
uso_mensual = np.array([np.random.normal(uso_base[s], 12) for s in segmento])
uso_mensual = np.clip(uso_mensual, 0, 200).round(1)

# Tickets de soporte: algunos clientes outliers con muchos tickets
tickets = np.random.exponential(scale=3, size=n).round().astype(int)
tickets[np.random.choice(n, 10)] = np.random.randint(20, 50, 10)  # outliers

# NPS: correlaciona inversamente con tickets y con churn
nps = np.clip(80 - tickets * 2 + np.random.normal(0, 10, n), 0, 100).round()

# Dias sin login: mayor inactividad -> mayor riesgo churn
dias_sin_login = np.random.exponential(scale=15, size=n).round().astype(int)
dias_sin_login = np.clip(dias_sin_login, 0, 180)

# Churn: funcion de inactividad, tickets y NPS bajo
prob_churn = 1 / (1 + np.exp(
    -(0.03 * dias_sin_login + 0.05 * tickets - 0.02 * nps - 1.5)
))
churn = (np.random.rand(n) < prob_churn).astype(int)

# Fecha ultima renovacion: 15%% de nulos (clientes cuyo ciclo no esta registrado)
fecha_renovacion = pd.date_range('2023-01-01', periods=n, freq='D')
fecha_renovacion = pd.Series(fecha_renovacion.astype(str))
idx_nulos = np.random.choice(n, int(n * 0.15), replace=False)
fecha_renovacion.iloc[idx_nulos] = None

# Valor contrato: Enterprise paga mas; 8%% de nulos (contratos no digitalizados)
valor_contrato = np.array([
    {'SMB': 500, 'Mid-Market': 2000, 'Enterprise': 8000}[s]
    + np.random.normal(0, 300)
    for s in segmento
]).round()
idx_nulos_vc = np.random.choice(n, int(n * 0.08), replace=False)
valor_contrato = pd.Series(valor_contrato.astype(float))
valor_contrato.iloc[idx_nulos_vc] = None

df = pd.DataFrame({
    'segmento': segmento,
    'uso_mensual': uso_mensual,
    'tickets_soporte': tickets,
    'nps': nps,
    'dias_sin_login': dias_sin_login,
    'fecha_renovacion': fecha_renovacion,
    'valor_contrato': valor_contrato,
    'churn': churn
})

df.to_csv('data/crm_clientes.csv', index=False)
print(f'Dataset generado: {len(df)} clientes, {df.churn.mean()*100:.1f}%% churn')
print(df.dtypes)
df.head()

Dataset generado: 300 clientes, 11.7%% churn
segmento             object
uso_mensual         float64
tickets_soporte       int64
nps                 float64
dias_sin_login        int64
fecha_renovacion     object
valor_contrato      float64
churn                 int64
dtype: object


,segmento,uso_mensual,tickets_soporte,nps,dias_sin_login,fecha_renovacion,valor_contrato,churn
0,SMB,20.5,1,75.0,30,2023-01-01,NaN,0
1,Enterprise,112.2,2,77.0,12,2023-01-02,7826.0,0
2,Mid-Market,80.7,2,73.0,12,2023-01-03,2249.0,0
3,Mid-Market,62.6,0,81.0,1,2023-01-04,1834.0,0
4,SMB,0.0,1,65.0,65,2023-01-05,697.0,1


## 2. Pregunta 1: ¿que datos faltan y que implica su ausencia?

Un mapa de nulos no es solo un control de calidad de datos.
Es una radiografia de que partes del negocio no estan bien instrumentadas.

**Lectura de negocio del mapa que vamos a generar:**
- `fecha_renovacion` nula: ~15%% de clientes sin ciclo de vida registrado.
  El modelo no habra visto estos clientes. Sus predicciones sobre ellos
  seran extrapolaciones sin garantia.
- `valor_contrato` nulo: ~8%% de contratos no digitalizados.
  Si el valor del contrato es predictivo del churn (contratos grandes
  tienen menos churn), ignorar estos clientes sesga el modelo hacia
  clientes de menor valor.

**La pregunta que hay que hacerse antes de imputar o eliminar nulos:**
¿El dato falta aleatoriamente, o falta exactamente en los casos mas dificiles
de predecir? Si los contratos no digitalizados son los mas antiguos o los
de mayor valor, eliminarlos es perder la informacion mas critica.

In [3]:
# Mapa de nulos: porcentaje por columna
nulos_pct = df.isnull().mean() * 100
nulos_pct = nulos_pct[nulos_pct > 0].sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Barra de porcentaje de nulos
ax = axes[0]
colors = ['#E53935' if p > 10 else '#FB8C00' for p in nulos_pct]
bars = ax.barh(nulos_pct.index, nulos_pct.values, color=colors)
for bar, pct in zip(bars, nulos_pct.values):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%%', va='center', fontsize=11)
ax.set_xlabel('Porcentaje de valores nulos', fontsize=11)
ax.set_title('Columnas con datos faltantes', fontsize=12)
ax.axvline(10, color='#E53935', linestyle='--', lw=1, alpha=0.6,
           label='Umbral critico (10%%)')
ax.legend(fontsize=9)
ax.set_xlim(0, 25)

# Heatmap de nulos por segmento
ax2 = axes[1]
nulos_seg = df.groupby('segmento')[['fecha_renovacion','valor_contrato']].apply(
    lambda x: x.isnull().mean() * 100
)
sns.heatmap(nulos_seg, annot=True, fmt='.1f', cmap='YlOrRd',
            ax=ax2, cbar_kws={'label': '%% nulos'}, vmin=0, vmax=30)
ax2.set_title('Nulos por segmento (%%)', fontsize=12)
ax2.set_xlabel('')

plt.suptitle('Mapa de datos faltantes - lectura de riesgo operativo',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('images/B02E_fig01.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: B02E_fig01.png')
print()
print('Interpretacion de negocio:')
print(f'  fecha_renovacion: {nulos_pct["fecha_renovacion"]:.1f}%% nulos -> '
      f'clientes sin ciclo de vida registrado')
print(f'  valor_contrato:   {nulos_pct["valor_contrato"]:.1f}%% nulos -> '
      f'contratos no digitalizados')

Guardado: B02E_fig01.png

Interpretacion de negocio:
  fecha_renovacion: 15.0%% nulos -> clientes sin ciclo de vida registrado
  valor_contrato:   8.0%% nulos -> contratos no digitalizados


## 3. Pregunta 2: ¿que variable importa para predecir el churn?

Una matriz de correlacion con el target responde 'que variable
observada se mueve junto con el churn'. No responde 'que causa el churn'.

**Lectura tecnica:** coeficiente de Pearson entre cada feature y churn.
Valores cercanos a 1 o -1 indican covariacion lineal fuerte.

**Lectura de negocio:**
- Si `dias_sin_login` tiene correlacion alta con churn, se puede actuar:
  contactar a clientes inactivos antes de que cancelen.
- Si `nps` tiene correlacion alta, la senal es previa: el cliente ya
  expreso insatisfaccion antes de cancelar. La intervencion debe ser
  antes de que el NPS baje, no despues.
- Una variable con correlacion baja puede seguir siendo util si
  es observable con antelacion suficiente para actuar.

In [4]:
# Correlacion de cada variable numerica con churn
numericas = ['uso_mensual', 'tickets_soporte', 'nps',
             'dias_sin_login', 'valor_contrato', 'churn']
df_num = df[numericas].copy()

corr_full = df_num.corr()
corr_target = corr_full['churn'].drop('churn').sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras: correlacion con churn
ax = axes[0]
colors = ['#E53935' if v > 0 else '#1565C0' for v in corr_target]
bars = ax.barh(corr_target.index, corr_target.values, color=colors)
for bar, val in zip(bars, corr_target.values):
    xpos = val + 0.01 if val >= 0 else val - 0.01
    ax.text(xpos, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=11)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Correlacion de Pearson con churn', fontsize=11)
ax.set_title('¿Que variable predice el churn?', fontsize=12)
from matplotlib.patches import Patch
leyenda = [
    Patch(facecolor='#E53935', label='Positiva: mas valor -> mas churn'),
    Patch(facecolor='#1565C0', label='Negativa: mas valor -> menos churn'),
]
ax.legend(handles=leyenda, fontsize=9, loc='lower right')

# Heatmap completo
ax2 = axes[1]
mask = np.triu(np.ones_like(corr_full, dtype=bool))
sns.heatmap(corr_full, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax2,
            annot_kws={'size': 9})
ax2.set_title('Matriz de correlacion completa\n'
              '(multicolinealidad entre features)', fontsize=12)

plt.tight_layout()
plt.savefig('images/B02E_fig02.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: B02E_fig02.png')
print()
print('Top variables correlacionadas con churn:')
print(corr_target.abs().sort_values(ascending=False).to_string())

Guardado: B02E_fig02.png

Top variables correlacionadas con churn:
tickets_soporte    0.260131
nps                0.247718
dias_sin_login     0.222338
valor_contrato     0.022838
uso_mensual        0.004218


## 4. ¿Quienes son nuestros clientes? - Histograma con segmentacion

**Pregunta de negocio:** ¿hay grupos de clientes con comportamientos
distintos mezclados en el mismo dataset?

**Lo que buscamos:**
- Distribuciones bimodales (dos picos): senalan dos poblaciones mezcladas
- Concentracion extrema en un valor: puede ser un valor por defecto incorrecto
- Asimetria fuerte: la media no representa bien al cliente tipico

**Accion si detectamos bimodalidad en `uso_mensual`:**
No hay que construir un modelo unico. Hay que segmentar primero
y modelar cada segmento por separado, o incluir el segmento como
variable explicativa.

In [5]:
# Histograma de uso_mensual por segmento
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Todos los clientes mezclados
ax = axes[0]
ax.hist(df['uso_mensual'], bins=25, color='#1565C0', alpha=0.7, edgecolor='white')
ax.axvline(df['uso_mensual'].mean(), color='#E53935', lw=2,
           label=f'Media: {df["uso_mensual"].mean():.1f}')
ax.axvline(df['uso_mensual'].median(), color='#F57F17', lw=2, linestyle='--',
           label=f'Mediana: {df["uso_mensual"].median():.1f}')
ax.set_xlabel('Uso mensual (sesiones)', fontsize=11)
ax.set_ylabel('Numero de clientes', fontsize=11)
ax.set_title('Todos los clientes mezclados\n'
             'La media no representa a nadie', fontsize=12)
ax.legend(fontsize=10)

# Por segmento: el origen de los picos
ax2 = axes[1]
colores = {'SMB': '#42A5F5', 'Mid-Market': '#66BB6A', 'Enterprise': '#EF5350'}
for seg, color in colores.items():
    datos = df[df['segmento'] == seg]['uso_mensual']
    ax2.hist(datos, bins=20, alpha=0.6, color=color,
             label=f'{seg} (n={len(datos)})', edgecolor='white')
ax2.set_xlabel('Uso mensual (sesiones)', fontsize=11)
ax2.set_ylabel('Numero de clientes', fontsize=11)
ax2.set_title('Desglosado por segmento\n'
              'Cada pico corresponde a una poblacion', fontsize=12)
ax2.legend(fontsize=10)

plt.suptitle('Histograma: identificar poblaciones mezcladas', fontsize=13)
plt.tight_layout()
plt.savefig('images/B02E_fig03.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: B02E_fig03.png')
print('Lectura: la distribucion total tiene forma multimodal.')
print('Un modelo entrenado sobre todos los datos confundira los tres segmentos.')

Guardado: B02E_fig03.png
Lectura: la distribucion total tiene forma multimodal.
Un modelo entrenado sobre todos los datos confundira los tres segmentos.


## 5. ¿Hay clientes atipicos o errores de datos? - Boxplot

**Pregunta de negocio:** ¿los outliers son errores de registro
o casos reales que el modelo deberia aprender?

**Lo que buscamos en `tickets_soporte`:**
- Outlier bajo = cliente nuevo o que no usa la plataforma (diferente a uno satisfecho)
- Outlier alto = cliente con problemas tecnicos graves O cliente que abusa del soporte
  como canal de comunicacion porque no tiene otro

**La decision critica:** antes de eliminar un outlier, responder:
¿este tipo de cliente existira en produccion? Si la respuesta es si,
el modelo debe haberlo visto durante el entrenamiento.

In [6]:
# Boxplot de tickets_soporte por segmento y churn
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Por segmento
ax = axes[0]
datos_box = [df[df['segmento'] == s]['tickets_soporte'].values
             for s in ['SMB', 'Mid-Market', 'Enterprise']]
bp = ax.boxplot(datos_box, labels=['SMB', 'Mid-Market', 'Enterprise'],
                patch_artist=True, notch=False,
                medianprops=dict(color='black', lw=2))
colores_box = ['#42A5F5', '#66BB6A', '#EF5350']
for patch, color in zip(bp['boxes'], colores_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel('Tickets de soporte', fontsize=11)
ax.set_title('Tickets por segmento\n'
             'Los puntos fuera del bigote son outliers', fontsize=12)
ax.set_xlabel('Segmento', fontsize=11)

# Por churn
ax2 = axes[1]
datos_churn = [df[df['churn'] == c]['tickets_soporte'].values
               for c in [0, 1]]
bp2 = ax2.boxplot(datos_churn, labels=['No churn', 'Churn'],
                  patch_artist=True,
                  medianprops=dict(color='black', lw=2))
for patch, color in zip(bp2['boxes'], ['#66BB6A', '#EF5350']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax2.set_ylabel('Tickets de soporte', fontsize=11)
ax2.set_title('Tickets por estado de churn\n'
              'La mediana es mas alta en clientes que cancelaron', fontsize=12)

# Anotar el outlier mas extremo
max_tickets = df['tickets_soporte'].max()
max_idx = df['tickets_soporte'].idxmax()
ax.annotate(f'Outlier: {max_tickets} tickets\n'
            f'(segmento: {df.loc[max_idx,"segmento"]})',
            xy=(df.loc[max_idx,'segmento'] == 'SMB' and 1 or
                df.loc[max_idx,'segmento'] == 'Mid-Market' and 2 or 3,
                max_tickets),
            xytext=(2.5, max_tickets - 5),
            fontsize=9, color='#B71C1C',
            arrowprops=dict(arrowstyle='->', color='#B71C1C'))

plt.suptitle('Boxplot: identificar outliers y decidir si son errores o casos reales',
             fontsize=13)
plt.tight_layout()
plt.savefig('images/B02E_fig04.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: B02E_fig04.png')
print(f'Cliente con mas tickets: {max_tickets} (segmento: {df.loc[max_idx,"segmento"]})')
print('Pregunta: ¿es este un error de registro o un cliente real?')

Guardado: B02E_fig04.png
Cliente con mas tickets: 49 (segmento: SMB)
Pregunta: ¿es este un error de registro o un cliente real?


## 6. ¿Como se relacionan dos KPIs? - Scatter con interpretacion de producto

**Pregunta de negocio:** ¿los clientes que mas usan la plataforma
son los mas satisfechos? ¿o hay clientes con uso alto pero NPS bajo?

**Lo que buscamos:**
- Correlacion positiva esperada: mas uso -> mas NPS (producto genera valor)
- Correlacion negativa: mas uso -> menos NPS (el producto tiene fricciones)
- Sin correlacion: uso y satisfaccion son independientes

**El hallazgo mas valioso en un scatter no es la correlacion:**
es el cuadrante inesperado. Clientes con uso alto y NPS bajo son clientes
atrapados en el producto, no satisfechos con el. Son un riesgo silencioso:
en cuanto aparezca un competidor, se van.

In [7]:
# Scatter: uso_mensual vs NPS, coloreado por churn
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_churn = {0: '#1565C0', 1: '#E53935'}
labels_churn = {0: 'No churn', 1: 'Churn'}

ax = axes[0]
for churn_val in [0, 1]:
    mask = df['churn'] == churn_val
    ax.scatter(df.loc[mask, 'uso_mensual'], df.loc[mask, 'nps'],
               c=colors_churn[churn_val], label=labels_churn[churn_val],
               alpha=0.5, s=40, edgecolors='none')

# Lineas de cuadrantes
ax.axvline(df['uso_mensual'].median(), color='gray', lw=1, linestyle='--', alpha=0.5)
ax.axhline(df['nps'].median(), color='gray', lw=1, linestyle='--', alpha=0.5)

# Anotar cuadrantes
ax.text(5, 95, 'Uso bajo\nNPS alto\n(satisfechos\nque no usan)',
        fontsize=8, color='#555', alpha=0.8)
ax.text(120, 95, 'Uso alto\nNPS alto\n(clientes\nideales)',
        fontsize=8, color='#1B5E20', alpha=0.9)
ax.text(120, 5, 'Uso alto\nNPS bajo\n(atrapados\nen el producto)',
        fontsize=8, color='#B71C1C', alpha=0.9)
ax.text(5, 5, 'Uso bajo\nNPS bajo\n(en riesgo\ninmediato)',
        fontsize=8, color='#E65100', alpha=0.9)

ax.set_xlabel('Uso mensual (sesiones)', fontsize=11)
ax.set_ylabel('NPS', fontsize=11)
ax.set_title('Uso vs NPS por estado de churn\n'
             'Cuadrante inferior-derecho: riesgo silencioso', fontsize=12)
ax.legend(fontsize=10)

# Scatter: dias_sin_login vs nps
ax2 = axes[1]
for churn_val in [0, 1]:
    mask = df['churn'] == churn_val
    ax2.scatter(df.loc[mask, 'dias_sin_login'], df.loc[mask, 'nps'],
                c=colors_churn[churn_val], label=labels_churn[churn_val],
                alpha=0.5, s=40, edgecolors='none')
ax2.set_xlabel('Dias sin login', fontsize=11)
ax2.set_ylabel('NPS', fontsize=11)
ax2.set_title('Inactividad vs NPS por estado de churn\n'
              'Clientes inactivos con NPS bajo: intervencion urgente', fontsize=12)
ax2.legend(fontsize=10)

plt.suptitle('Scatter: detectar contradicciones de producto en los datos',
             fontsize=13)
plt.tight_layout()
plt.savefig('images/B02E_fig05.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: B02E_fig05.png')

# Contar clientes en cuadrante de riesgo silencioso
riesgo = df[
    (df['uso_mensual'] > df['uso_mensual'].median()) &
    (df['nps'] < df['nps'].median())
]
print(f'Clientes en cuadrante de riesgo silencioso (uso alto + NPS bajo): {len(riesgo)}')
print(f'  De esos, {riesgo["churn"].sum()} cancelaron ({riesgo["churn"].mean()*100:.1f}%%)')

Guardado: B02E_fig05.png
Clientes en cuadrante de riesgo silencioso (uso alto + NPS bajo): 77
  De esos, 10 cancelaron (13.0%%)


## 7. Ejercicio tecnico

**Enunciado:**
Con el dataset `df` que hemos generado:

1. Calcula la tasa de churn por segmento. ¿En que segmento es mas alta?
2. Identifica la variable numerica con mayor correlacion absoluta con `churn`.
3. Hay clientes con `dias_sin_login > 60` que NO cancelaron.
   ¿Cuantos son? ¿Que segmento representan mayoritariamente?

**Objetivo:** practicar EDA orientado a preguntas, no a descripcion.

In [8]:
# TODO: responde las tres preguntas
# ── 1. Tasa de churn por segmento ──────────────────────────────────────
churn_segmento = None  # df.groupby('segmento')['churn'].mean()
print('Tasa de churn por segmento:')
print(churn_segmento)

# ── 2. Variable con mayor correlacion con churn ──────────────────────────
numericas = ['uso_mensual', 'tickets_soporte', 'nps', 'dias_sin_login']
corr_churn = None  # df[numericas + ['churn']].corr()['churn'].drop('churn')
print('\nCorrelacion con churn:')
print(corr_churn)

# ── 3. Inactivos que no cancelaron ──────────────────────────────────────
inactivos_ok = None  # df[(df['dias_sin_login'] > 60) & (df['churn'] == 0)]
print(f'\nClientes inactivos (>60 dias) que no cancelaron: {len(inactivos_ok) if inactivos_ok is not None else "?"}')

Tasa de churn por segmento:
None

Correlacion con churn:
None

Clientes inactivos (>60 dias) que no cancelaron: ?


In [9]:
# SOLUCION - ejecutar tras intentarlo
# 1. Tasa de churn por segmento
churn_seg = df.groupby('segmento')['churn'].mean().sort_values(ascending=False)
print('Tasa de churn por segmento:')
print(churn_seg.to_string())
print(f'  Segmento con mas churn: {churn_seg.idxmax()} ({churn_seg.max()*100:.1f}%%)')

# 2. Variable mas correlacionada
corr_churn = df[['uso_mensual','tickets_soporte','nps','dias_sin_login','churn']]\
               .corr()['churn'].drop('churn').abs().sort_values(ascending=False)
print(f'\nVariable mas correlacionada con churn: {corr_churn.idxmax()} '
      f'(r={corr_churn.max():.3f})')

# 3. Inactivos que no cancelaron
inactivos_ok = df[(df['dias_sin_login'] > 60) & (df['churn'] == 0)]
seg_freq = inactivos_ok['segmento'].value_counts()
print(f'\nClientes inactivos (>60 dias) sin churn: {len(inactivos_ok)}')
print(f'  Segmento mayoritario: {seg_freq.idxmax()} ({seg_freq.max()} clientes)')
print('  Interpretacion: inactividad no implica churn en todos los segmentos.')
print('  Puede haber clientes de bajo uso que son estables a largo plazo.')

Tasa de churn por segmento:
segmento
Enterprise    0.145833
SMB           0.127273
Mid-Market    0.080460
  Segmento con mas churn: Enterprise (14.6%%)

Variable mas correlacionada con churn: tickets_soporte (r=0.260)

Clientes inactivos (>60 dias) sin churn: 2
  Segmento mayoritario: SMB (2 clientes)
  Interpretacion: inactividad no implica churn en todos los segmentos.
  Puede haber clientes de bajo uso que son estables a largo plazo.


## 8. Ejercicio de decision

**Caso:**
El equipo de Customer Success tiene capacidad para contactar proactivamente
a 30 clientes este mes (de 300 totales).

Con el EDA que acabas de hacer, responde:

1. ¿Que 30 clientes priorizarias y en base a que variables?
2. ¿Construirias un modelo de churn con los datos tal como estan,
   o primero tomarias alguna accion sobre los datos?
3. ¿Hay alguna variable que no usarias como feature de un modelo
   aunque tenga alta correlacion con churn? ¿Por que?

**Criterio de evaluacion:**
Una respuesta solida menciona:
- Priorizar clientes en cuadrante de riesgo: dias_sin_login alto + nps bajo
- No usar `fecha_renovacion` como feature sin imputar o sin entender
  por que falta en esos clientes especificos
- No usar `valor_contrato` directamente si los nulos coinciden con los
  contratos mas criticos (sesgo de seleccion)
- Reconocer que `nps` es observable antes del churn, lo que lo hace
  accionable, a diferencia de `uso_mensual` que es contemporaneo

## 9. Assets generados

| Fichero | Contenido |
|---|---|
| `data/crm_clientes.csv` | Dataset CRM simulado (300 clientes, 8 variables) |
| `images/B02E_fig01.png` | Mapa de nulos: porcentaje por columna y por segmento |
| `images/B02E_fig02.png` | Correlacion con churn: barras + heatmap completo |
| `images/B02E_fig03.png` | Histograma uso_mensual: total vs por segmento |
| `images/B02E_fig04.png` | Boxplot tickets_soporte: por segmento y por churn |
| `images/B02E_fig05.png` | Scatter uso/NPS y dias_sin_login/NPS por churn |

**Dependencias:** numpy, pandas, matplotlib, seaborn